# OOF residual correction components for v007

Definition-only components. Loading the code cells reads no assignment data, creates no files, and fits no model. Load the `v005-feature-definition` cells from `feature_components_v005.ipynb` first. The correction uses its 47 `combined` known-input features (including R, C, step, and inspiratory phase) plus the native base prediction. Row and breath IDs are alignment/provenance keys, never model inputs.

The only production model is a fixed LightGBM MAE regressor: 500 trees, 127 leaves, learning rate .03, minimum 200 samples per leaf. It learns `approved pressure - genuinely OOF base pressure` on inspired rows only, with equal weight per inspired row. There is no validation argument, early stopping, scaling, target normalization, or hyperparameter search. Original A validation breaths must be explicitly forbidden in every base fit, base preprocessing fit, and calibration fit. The OOF base schedule is frozen before OOF construction; using the already A-selected compact schedule does **not** turn these folds into independent model-selection evidence.

The manifest records actual caller-attested base fit/preprocessing groups, their hashes, the per-breath held-out fold, canonical row IDs, and the frozen base recipe. It rejects in-sample residuals whenever the supplied provenance reveals them; provenance cannot independently prove that a caller trained the declared checkpoint. Record checkpoint paths/hashes from the actual fits. Build OOF predictions only from held-out fold checkpoints. Never replace them with predictions from the ordinary all-Afit base checkpoint.

Use one fixed calibration model. Compare predeclared correction strengths `0`, `.5`, and `1` on original A only, then freeze that choice before any B confirmation. A-selected shrinkage and the new calibration model can overfit A; an improvement needs independent frozen confirmation. For the fixed equal ensemble, average deep native pressure with the corrected compact native pressure. Expiration predictions pass through unchanged.

Interfaces: `residual_group_folds_v007`, `residual_recipe_sha256_v007`, `make_residual_oof_manifest_v007`, `validate_residual_oof_v007`, `fit_residual_v007`, `predict_residual_corrected_v007`, `save_residual_v007`, and `load_residual_v007`. Prediction inputs are DataFrames with exactly `id, base_pressure` in canonical frame row order. `predict_residual_corrected_v007` returns `[breaths, 80]` native pressures. LightGBM is imported only when production fitting is explicitly called. The synthetic test double exercises the same core fitting path without pretending to validate an unavailable LightGBM installation.

`make_residual_synthetic_fixture_v007()` exposes generated inputs and valid fold attestations. `run_residual_lightgbm_smoke_v007(n_jobs=2)` explicitly exercises the actual fixed LightGBM fit, prediction, and save/load path on 27 synthetic breaths in an existing environment. It never runs automatically or installs dependencies.


In [ ]:
import copy
import hashlib
import json
import os
import pickle
import tempfile
import uuid
from pathlib import Path

import numpy as np
import pandas as pd

V007_RESIDUAL_SCHEMA = 'oof-native-residual-v007'
V007_RESIDUAL_STEPS = 80
V007_RESIDUAL_GAMMAS = (0.0, 0.5, 1.0)
V007_RESIDUAL_MODEL_DEFAULTS = {
    'objective': 'regression_l1', 'n_estimators': 500,
    'learning_rate': 0.03, 'num_leaves': 127,
    'min_child_samples': 200, 'max_depth': -1,
    'reg_lambda': 1.0, 'reg_alpha': 0.0,
    'subsample': 1.0, 'colsample_bytree': 1.0,
    'random_state': 707, 'n_jobs': 8, 'verbosity': -1,
    'deterministic': True, 'force_col_wise': True,
}
V007_RESIDUAL_KNOWN_COLUMNS = (
    'id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out',
)


def _residual_json_hash_v007(value):
    serialized = json.dumps(value, sort_keys=True, separators=(',', ':'),
                            allow_nan=False).encode('utf-8')
    return hashlib.sha256(serialized).hexdigest()


def _residual_integer_ids_v007(values, name, unique=True):
    raw = np.asarray(values)
    if raw.ndim != 1 or not len(raw):
        raise ValueError(f'{name} must be a nonempty one-dimensional ID sequence')
    if not np.issubdtype(raw.dtype, np.number) or not np.isfinite(raw).all():
        raise ValueError(f'{name} must contain finite numeric integer IDs')
    if not np.all(raw == np.floor(raw)) or np.any(raw < 0):
        raise ValueError(f'{name} must contain nonnegative integer IDs')
    # Competition IDs fit int64. Reject unsafe casts before hashing/comparison.
    if np.any(raw.astype(np.float64) >= float(2**63)):
        raise ValueError(f'{name} exceeds signed int64 range')
    result = raw.astype(np.int64)
    if unique and len(np.unique(result)) != len(result):
        raise ValueError(f'{name} contains duplicates')
    return result


def residual_group_sha256_v007(values):
    ids = _residual_integer_ids_v007(values, 'breath IDs')
    return hashlib.sha256(np.sort(ids).astype('<i8').tobytes()).hexdigest()


def _residual_row_sha256_v007(values):
    ids = _residual_integer_ids_v007(values, 'row IDs')
    return hashlib.sha256(ids.astype('<i8').tobytes()).hexdigest()


def _residual_frame_groups_v007(frame):
    if not callable(globals().get('_validate_feature_frame_v005')):
        raise RuntimeError('Load v005-feature-definition cells before residual components')
    if not isinstance(frame, pd.DataFrame):
        raise TypeError('Expected a pandas DataFrame')
    # This selection deliberately never reads pressure or any unapproved column.
    if not frame.columns.is_unique:
        raise ValueError('Duplicate frame columns')
    missing = set(V007_RESIDUAL_KNOWN_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing known input columns: {sorted(missing)}')
    known = frame.loc[:, list(V007_RESIDUAL_KNOWN_COLUMNS)]
    groups = _validate_feature_frame_v005(known)
    return _residual_integer_ids_v007(groups, 'frame breath IDs')


def _residual_base_vector_v007(frame, predictions):
    if not isinstance(predictions, pd.DataFrame):
        raise TypeError('Base predictions must be a DataFrame with explicit id/base_pressure')
    if list(predictions.columns) != ['id', 'base_pressure']:
        raise ValueError('Base prediction columns must be exactly [id, base_pressure]')
    row_ids = _residual_integer_ids_v007(frame['id'].to_numpy(), 'frame row IDs')
    prediction_ids = _residual_integer_ids_v007(predictions['id'].to_numpy(), 'prediction row IDs')
    if not np.array_equal(row_ids, prediction_ids):
        raise ValueError('Base prediction row IDs/order do not match canonical frame rows')
    base = predictions['base_pressure'].to_numpy(dtype=np.float64)
    if base.shape != (len(frame),) or not np.isfinite(base).all():
        raise ValueError('Base pressures must be finite native values for every row')
    return base


def residual_group_folds_v007(breath_ids, n_splits=3, seed=707):
    """Balanced whole-breath folds, invariant to caller group order."""
    groups = _residual_integer_ids_v007(breath_ids, 'breath IDs')
    if isinstance(n_splits, bool) or int(n_splits) != n_splits:
        raise ValueError('n_splits must be an integer')
    n_splits = int(n_splits)
    if n_splits < 2 or len(groups) < n_splits:
        raise ValueError('Need at least two folds and one breath per fold')
    ordered = np.sort(groups)
    shuffled = np.random.default_rng(int(seed)).permutation(ordered)
    group_to_fold = {int(g): int(i % n_splits) for i, g in enumerate(shuffled)}
    return np.asarray([group_to_fold[int(g)] for g in groups], dtype=np.int64)


def residual_recipe_sha256_v007(base_recipe):
    """Audit a frozen A-selected recipe; this does not train or select anything."""
    if not isinstance(base_recipe, dict):
        raise TypeError('base_recipe must be a dictionary')
    required = {'recipe_id', 'feature_mode', 'selected_epochs', 'learning_rates',
                'selection_source', 'frozen_before_oof', 'target_units'}
    if not required.issubset(base_recipe):
        raise ValueError(f'Missing frozen recipe fields: {sorted(required-set(base_recipe))}')
    if not isinstance(base_recipe['recipe_id'], str) or not base_recipe['recipe_id']:
        raise ValueError('recipe_id must identify the frozen base model')
    if base_recipe['feature_mode'] not in ('causal20', 'offline', 'dynamics', 'combined'):
        raise ValueError('Unknown base feature mode')
    epochs = base_recipe['selected_epochs']
    if isinstance(epochs, bool) or int(epochs) != epochs or epochs <= 0:
        raise ValueError('selected_epochs must be a positive integer')
    rates = np.asarray(base_recipe['learning_rates'], dtype=np.float64)
    if rates.shape != (int(epochs),) or not np.isfinite(rates).all() or np.any(rates <= 0):
        raise ValueError('Frozen learning_rates must contain one positive rate per epoch')
    if base_recipe['selection_source'] != 'original_A':
        raise ValueError('Only the preselected original_A recipe is approved')
    if base_recipe['frozen_before_oof'] is not True:
        raise ValueError('Base recipe must be frozen before OOF construction')
    if base_recipe['target_units'] != 'native_pressure':
        raise ValueError('Base predictions must be in native pressure units')
    return _residual_json_hash_v007(base_recipe)


In [ ]:
def _residual_expected_groups_v007(frame, approved_fit_breath_ids, forbidden_breath_ids):
    groups = _residual_frame_groups_v007(frame)
    approved = _residual_integer_ids_v007(approved_fit_breath_ids, 'approved fit breath IDs')
    forbidden = _residual_integer_ids_v007(forbidden_breath_ids, 'forbidden validation breath IDs')
    if np.intersect1d(approved, forbidden).size:
        raise ValueError('Approved fit groups overlap forbidden validation groups')
    if not np.array_equal(np.sort(groups), np.sort(approved)):
        raise ValueError('Calibration frame must contain exactly the approved fit breaths')
    if np.intersect1d(groups, forbidden).size:
        raise ValueError('Forbidden validation breaths reached calibration fitting')
    return groups, approved, forbidden


def make_residual_oof_manifest_v007(frame, fold_ids, fit_records,
                                   approved_fit_breath_ids, forbidden_breath_ids,
                                   base_recipe):
    """Validate actual base-fit attestations and record explicit OOF provenance.

    fold_ids contains one integer per canonical frame breath. Each fit record
    requires fold_id, fit_breath_ids, preprocessing_fit_breath_ids, and
    base_recipe_sha256. Both group lists must equal the other folds exactly.
    Optional checkpoint_path/checkpoint_sha256 identify the actual saved base fit.
    """
    groups, approved, forbidden = _residual_expected_groups_v007(
        frame, approved_fit_breath_ids, forbidden_breath_ids)
    recipe_hash = residual_recipe_sha256_v007(base_recipe)
    folds = _residual_integer_ids_v007(fold_ids, 'per-breath folds', unique=False)
    if folds.shape != groups.shape:
        raise ValueError('Expected one fold ID per breath')
    unique_folds = np.unique(folds)
    if len(unique_folds) < 2 or not np.array_equal(unique_folds, np.arange(len(unique_folds))):
        raise ValueError('OOF folds must be consecutive 0..K-1 with K >= 2')
    records = list(fit_records)
    if len(records) != len(unique_folds):
        raise ValueError('Need exactly one actual fit record per OOF fold')
    by_fold = {}
    normalized = []
    for record in records:
        if not isinstance(record, dict):
            raise TypeError('Each base fit record must be a dictionary')
        required = {'fold_id', 'fit_breath_ids', 'preprocessing_fit_breath_ids', 'base_recipe_sha256'}
        if not required.issubset(record):
            raise ValueError(f'Missing actual base fit fields: {sorted(required-set(record))}')
        raw_fold_id = record['fold_id']
        if isinstance(raw_fold_id, bool) or int(raw_fold_id) != raw_fold_id:
            raise ValueError('Fit record fold_id must be an integer')
        fold_id = int(raw_fold_id)
        if fold_id not in unique_folds or fold_id in by_fold:
            raise ValueError('Unknown or duplicate base fit fold')
        by_fold[fold_id] = record
        expected_fit = np.sort(groups[folds != fold_id])
        heldout = groups[folds == fold_id]
        fit_ids = _residual_integer_ids_v007(record['fit_breath_ids'], 'actual base fit groups')
        stats_ids = _residual_integer_ids_v007(record['preprocessing_fit_breath_ids'], 'actual base preprocessing fit groups')
        for actual, name in ((fit_ids, 'base fit'), (stats_ids, 'base preprocessing fit')):
            if np.intersect1d(actual, heldout).size or np.intersect1d(actual, forbidden).size:
                raise ValueError(f'OOF leakage: {name} includes heldout/forbidden breaths')
            if not np.array_equal(np.sort(actual), expected_fit):
                raise ValueError(f'{name} groups must equal the exact held-out-fold complement')
        if record['base_recipe_sha256'] != recipe_hash:
            raise ValueError('OOF base fit does not use the frozen recipe hash')
        item = {
            'fold_id': fold_id,
            'fit_breath_ids': np.sort(fit_ids).tolist(),
            'preprocessing_fit_breath_ids': np.sort(stats_ids).tolist(),
            'heldout_breath_ids': heldout.tolist(),
            'fit_group_sha256': residual_group_sha256_v007(fit_ids),
            'preprocessing_fit_group_sha256': residual_group_sha256_v007(stats_ids),
            'heldout_group_sha256': residual_group_sha256_v007(heldout),
            'base_recipe_sha256': recipe_hash,
        }
        for field in ('checkpoint_path', 'checkpoint_sha256'):
            if field in record:
                if not isinstance(record[field], str) or not record[field]:
                    raise ValueError(f'{field} must be a nonempty string when supplied')
                item[field] = record[field]
        normalized.append(item)
    manifest = {
        'schema': V007_RESIDUAL_SCHEMA,
        'purpose': 'original_Afit_only_OOF_native_residuals',
        'breath_ids': groups.tolist(), 'fold_ids': folds.tolist(),
        'row_count': int(len(frame)),
        'canonical_row_id_sha256': _residual_row_sha256_v007(frame['id'].to_numpy()),
        'approved_fit_group_sha256': residual_group_sha256_v007(approved),
        'forbidden_group_sha256': residual_group_sha256_v007(forbidden),
        'base_recipe': copy.deepcopy(base_recipe), 'base_recipe_sha256': recipe_hash,
        'fold_records': sorted(normalized, key=lambda item: item['fold_id']),
        'provenance_limit': 'Caller-attested actual fit groups; checkpoint training is not independently replayed.',
    }
    manifest['manifest_sha256'] = _residual_json_hash_v007(manifest)
    return manifest


def validate_residual_oof_v007(frame, oof_predictions_df, manifest,
                               approved_fit_breath_ids, forbidden_breath_ids):
    """Rebuild all group/recipe proofs and bind predictions to canonical row IDs."""
    if not isinstance(manifest, dict) or manifest.get('schema') != V007_RESIDUAL_SCHEMA:
        raise ValueError('Unknown residual OOF manifest schema')
    rebuilt = make_residual_oof_manifest_v007(
        frame, manifest.get('fold_ids', []), manifest.get('fold_records', []),
        approved_fit_breath_ids, forbidden_breath_ids, manifest.get('base_recipe'))
    if rebuilt != manifest:
        raise ValueError('OOF manifest differs from reconstructed canonical proof')
    base = _residual_base_vector_v007(frame, oof_predictions_df)
    return {
        'manifest_sha256': manifest['manifest_sha256'],
        'base_prediction_sha256': hashlib.sha256(base.astype('<f8').tobytes()).hexdigest(),
        'row_count': int(len(frame)), 'breath_count': int(len(manifest['breath_ids'])),
        'fold_count': int(len(manifest['fold_records'])),
        'OOF_exclusion_verified': True,
        'original_A_validation_excluded': True,
    }


In [ ]:
def _residual_model_configuration_v007(model_cfg=None):
    supplied = {} if model_cfg is None else dict(model_cfg)
    # Runtime controls only: capacity/loss/schedule are a single fixed recipe.
    unknown = set(supplied) - {'n_jobs', 'random_state'}
    if unknown:
        raise ValueError(f'Only n_jobs/random_state runtime overrides are allowed: {sorted(unknown)}')
    cfg = dict(V007_RESIDUAL_MODEL_DEFAULTS, **supplied)
    for name in ('n_jobs', 'random_state'):
        value = cfg[name]
        if isinstance(value, bool) or int(value) != value:
            raise ValueError(f'{name} must be an integer')
        cfg[name] = int(value)
    if cfg['n_jobs'] < 1 or cfg['random_state'] < 0:
        raise ValueError('Use positive n_jobs and nonnegative random_state')
    return cfg


def _residual_feature_names_v007():
    if not callable(globals().get('feature_names_v005')):
        raise RuntimeError('Load v005-feature-definition cells first')
    names = list(feature_names_v005('combined'))
    if len(names) != 47 or len(set(names)) != len(names):
        raise ValueError('Expected the exact v005 combined 47-feature schema')
    return names + ['base_pressure_native']


def _residual_feature_matrix_v007(frame, base, inspired_only=True, chunk_breaths=2048):
    """Known inputs only, built by whole-breath chunks to bound temporary memory."""
    if isinstance(chunk_breaths, bool) or int(chunk_breaths) != chunk_breaths or chunk_breaths < 1:
        raise ValueError('chunk_breaths must be a positive integer')
    names = _residual_feature_names_v007()
    inspiration = frame['u_out'].to_numpy() == 0
    keep = inspiration if inspired_only else np.ones(len(frame), dtype=bool)
    matrix = np.empty((int(keep.sum()), len(names)), dtype=np.float32)
    cursor = 0
    chunk_rows = int(chunk_breaths)*V007_RESIDUAL_STEPS
    for start in range(0, len(frame), chunk_rows):
        stop = min(start+chunk_rows, len(frame))
        known = frame.iloc[start:stop].loc[:, list(V007_RESIDUAL_KNOWN_COLUMNS)]
        x, feature_names, _ = make_sequence_features(known, 'combined')
        if list(feature_names) != names[:-1]:
            raise ValueError('Residual feature schema changed within an inference call')
        selected = keep[start:stop]
        count = int(selected.sum())
        matrix[cursor:cursor+count, :-1] = x.reshape(-1, len(feature_names))[selected]
        matrix[cursor:cursor+count, -1] = base[start:stop][selected]
        cursor += count
    if cursor != len(matrix) or not np.isfinite(matrix).all():
        raise ValueError('Nonfinite/incorrectly sized residual features')
    return matrix, names, keep


def _fit_residual_core_v007(frame, oof_predictions_df, manifest,
                            approved_fit_breath_ids, forbidden_breath_ids,
                            estimator, model_cfg, model_kind):
    audit = validate_residual_oof_v007(
        frame, oof_predictions_df, manifest, approved_fit_breath_ids, forbidden_breath_ids)
    if 'pressure' not in frame.columns or not pd.api.types.is_numeric_dtype(frame['pressure']):
        raise ValueError('Calibration fitting requires approved numeric pressure labels')
    base = _residual_base_vector_v007(frame, oof_predictions_df)
    x, feature_names, inspiration = _residual_feature_matrix_v007(frame, base)
    # Do not access expiration labels, even for diagnostics or finite checks.
    pressure = frame.loc[inspiration, 'pressure'].to_numpy(dtype=np.float64)
    if not len(pressure) or not np.isfinite(pressure).all():
        raise ValueError('Approved inspired labels must be finite')
    residual = pressure-base[inspiration]
    if not np.isfinite(residual).all():
        raise ValueError('Native OOF residuals must be finite')
    # Equal point weights exactly match inspired-row MAE; no per-breath reweighting.
    estimator.fit(x, residual)
    metadata = {
        'schema': V007_RESIDUAL_SCHEMA, 'model_kind': model_kind,
        'feature_mode': 'combined', 'feature_names': feature_names,
        'model_cfg': copy.deepcopy(model_cfg),
        'fit_scope': 'approved_original_Afit_only', 'fit_target': 'native_pressure_minus_OOF_base_pressure',
        'fit_rows': int(len(residual)), 'all_rows': int(len(frame)),
        'sample_weighting': 'uniform_per_inspired_row; expiration excluded',
        'target_units': 'native_pressure', 'expiration_policy': 'unchanged_base_prediction',
        'allowed_gammas': list(V007_RESIDUAL_GAMMAS),
        'validation_arrays_accepted': False, 'early_stopping': False,
        'OOF_audit': audit, 'OOF_manifest': copy.deepcopy(manifest),
        'inspired_residual_sha256': hashlib.sha256(residual.astype('<f8').tobytes()).hexdigest(),
    }
    metadata['metadata_sha256'] = _residual_json_hash_v007(metadata)
    return {'model': estimator, 'metadata': metadata}


def fit_residual_v007(frame, oof_predictions_df, manifest,
                      approved_fit_breath_ids, forbidden_breath_ids, model_cfg=None):
    """Fit one fixed calibration model. No validation labels/arrays are accepted.

    Only actual OOF base predictions on approved original Afit breaths may be
    supplied. forbidden_breath_ids must explicitly include original A validation.
    """
    # Fail leakage/alignment checks before touching optional dependencies.
    validate_residual_oof_v007(frame, oof_predictions_df, manifest,
                              approved_fit_breath_ids, forbidden_breath_ids)
    cfg = _residual_model_configuration_v007(model_cfg)
    try:
        from lightgbm import LGBMRegressor
    except ImportError as exc:
        raise RuntimeError('Production residual fitting requires the existing course LightGBM environment; no automatic installation occurs') from exc
    estimator = LGBMRegressor(**cfg)
    return _fit_residual_core_v007(
        frame, oof_predictions_df, manifest, approved_fit_breath_ids,
        forbidden_breath_ids, estimator, cfg, 'lightgbm_fixed_MAE_127leaves_500trees')


def _validate_residual_bundle_v007(bundle):
    if not isinstance(bundle, dict) or set(bundle) != {'model', 'metadata'}:
        raise ValueError('Invalid residual bundle')
    metadata = bundle['metadata']
    if not isinstance(metadata, dict) or metadata.get('schema') != V007_RESIDUAL_SCHEMA:
        raise ValueError('Invalid residual metadata schema')
    proof = dict(metadata)
    recorded_hash = proof.pop('metadata_sha256', None)
    if recorded_hash != _residual_json_hash_v007(proof):
        raise ValueError('Residual metadata integrity check failed')
    if metadata.get('feature_names') != _residual_feature_names_v007():
        raise ValueError('Residual feature names differ from current combined47 schema')
    if metadata.get('target_units') != 'native_pressure' or not callable(getattr(bundle['model'], 'predict', None)):
        raise ValueError('Residual model must predict native pressure errors')
    return metadata


def predict_residual_corrected_v007(bundle, frame, base_predictions_df, gamma=1.0,
                                     chunk_breaths=2048):
    """Return base + gamma*predicted error on inspiration, native [breaths,80].

    frame pressure labels, if present, are ignored. The ordinary all-Afit compact
    checkpoint can supply the base predictions for original A or later inference.
    """
    metadata = _validate_residual_bundle_v007(bundle)
    _residual_frame_groups_v007(frame)
    if isinstance(gamma, bool) or float(gamma) not in V007_RESIDUAL_GAMMAS:
        raise ValueError(f'gamma must be predeclared in {V007_RESIDUAL_GAMMAS}')
    gamma = float(gamma)
    base = _residual_base_vector_v007(frame, base_predictions_df)
    corrected = base.copy()
    if gamma == 0.0:
        return corrected.reshape(-1, V007_RESIDUAL_STEPS)
    if isinstance(chunk_breaths, bool) or int(chunk_breaths) != chunk_breaths or chunk_breaths < 1:
        raise ValueError('chunk_breaths must be a positive integer')
    chunk_rows = int(chunk_breaths)*V007_RESIDUAL_STEPS
    for start in range(0, len(frame), chunk_rows):
        stop = min(start+chunk_rows, len(frame))
        chunk = frame.iloc[start:stop]
        x, names, keep = _residual_feature_matrix_v007(
            chunk, base[start:stop], inspired_only=True, chunk_breaths=chunk_breaths)
        if names != metadata['feature_names']:
            raise ValueError('Saved residual model feature schema changed')
        if len(x):
            delta = np.asarray(bundle['model'].predict(x), dtype=np.float64)
            if delta.shape != (int(keep.sum()),) or not np.isfinite(delta).all():
                raise ValueError('Residual model returned nonfinite/wrong-sized native errors')
            block = corrected[start:stop]
            block[keep] += gamma*delta
    if not np.isfinite(corrected).all():
        raise ValueError('Corrected native pressure overflowed')
    return corrected.reshape(-1, V007_RESIDUAL_STEPS)


def save_residual_v007(bundle, path, overwrite=False):
    """Save a trusted local pickle atomically; return its SHA256 for later loads."""
    _validate_residual_bundle_v007(bundle)
    target = Path(path)
    if target.exists() and not overwrite:
        raise FileExistsError(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name+'.tmp-'+uuid.uuid4().hex)
    try:
        with temporary.open('wb') as stream:
            pickle.dump(bundle, stream, protocol=pickle.HIGHEST_PROTOCOL)
        digest = hashlib.sha256(temporary.read_bytes()).hexdigest()
        os.replace(temporary, target)
    finally:
        if temporary.exists():
            temporary.unlink()
    return {'path': str(target), 'sha256': digest,
            'metadata_sha256': bundle['metadata']['metadata_sha256']}


def load_residual_v007(path, expected_sha256=None):
    """Load only a trusted, locally created residual artifact; pickle executes code."""
    raw = Path(path).read_bytes()
    if expected_sha256 is not None and hashlib.sha256(raw).hexdigest() != expected_sha256:
        raise ValueError('Residual artifact SHA256 mismatch')
    bundle = pickle.loads(raw)
    _validate_residual_bundle_v007(bundle)
    return bundle


## Caller sequence (illustrative; never executed automatically)

1. Define the original Afit and Avalidation breath groups from the existing split. Obtain order-invariant fold IDs on Afit. Freeze the original-A compact schedule before constructing folds.
2. For each fold, fit a fresh compact model and all target/input preprocessing on the exact other-fold groups. Predict only its held-out groups; place predictions in the canonical Afit row IDs. Record actual model-fit and preprocessing-fit IDs and `residual_recipe_sha256_v007(base_recipe)`; retain checkpoint paths and hashes.
3. Build the manifest with those records, Afit-approved IDs, and the explicitly forbidden original Avalidation IDs. Create `oof_predictions_df = pd.DataFrame({'id': Afit_frame['id'].to_numpy(), 'base_pressure': oof_native_predictions.reshape(-1)})`.
4. Explicitly call `fit_residual_v007` once. The function trains only the OOF native residual target on inspired Afit rows. Save its artifact and SHA256 receipt. Original Avalidation labels must remain outside this call.
5. Query the ordinary original-A compact checkpoint, wrap predictions with the actual canonical row IDs, and call `predict_residual_corrected_v007` for the predeclared strengths. The base checkpoint used for query predictions is ordinarily trained on all Afit, unlike the OOF calibration predictions. Freeze any A-selected strength before B. A fixed ensemble is `.5 * deep_native + .5 * corrected_compact_native`.

`run_residual_component_checks_v007()` is an explicit synthetic-only test call. It writes only temporary toy artifacts in its own temporary directory and returns a report; callers decide whether and where to persist that report. Saved pickle artifacts must come from a trusted source. A recorded fit-group hash is an audit trail, not cryptographic proof of the training execution.


In [ ]:
class _SyntheticMedianResidualV007:
    """NumPy-only test double; never a production model or fallback."""
    def fit(self, x, residual):
        self.n_fit_rows_ = int(len(residual))
        self.target_ = np.asarray(residual, dtype=np.float64).copy()
        self.value_ = float(np.median(self.target_))
        self.n_features_in_ = int(x.shape[1])
        return self

    def predict(self, x):
        if x.shape[1] != self.n_features_in_:
            raise ValueError('Synthetic model feature mismatch')
        return np.full(len(x), self.value_, dtype=np.float64)


def make_residual_synthetic_fixture_v007(n_breaths=9, variable_residual=False):
    """Generate CPU-only inputs/labels and valid fold attestations, never read data."""
    if isinstance(n_breaths, bool) or int(n_breaths) != n_breaths or n_breaths < 3:
        raise ValueError('Synthetic fixture needs at least three whole breaths')
    n_breaths = int(n_breaths)
    breath_ids = 101 + 202*np.arange(n_breaths, dtype=np.int64)
    frames = []
    for i, group in enumerate(breath_ids):
        step = np.arange(80)
        t = step*.03 + .0001*(i % 9)*step
        u = 8.0 + i % 9 + 10*np.sin(step/30.)
        out = (step >= 27+i % 9).astype(np.int64)
        base = 5.0 + .12*u + .03*step + .2*(i % 9)
        residual = 2.0 + .2*(i % 3) + .015*step if variable_residual else np.full(80, 2.0)
        pressure = np.where(out == 0, base+residual, np.nan)
        frames.append(pd.DataFrame({'id': 10000+i*80+step,
            'breath_id': group, 'R': (5,20,50)[i % 3], 'C': (10,20,50)[(i//3) % 3],
            'time_step': t, 'u_in': u, 'u_out': out, 'pressure': pressure,
            'synthetic_base': base}))
    frame = pd.concat(frames, ignore_index=True)
    predictions = pd.DataFrame({'id': frame['id'], 'base_pressure': frame['synthetic_base']})
    forbidden = np.asarray([999001, 999002])
    if np.intersect1d(forbidden, breath_ids).size:
        raise ValueError('Synthetic group IDs unexpectedly overlap forbidden fixture IDs')
    folds = residual_group_folds_v007(breath_ids)
    recipe = {'recipe_id': 'synthetic-fixed-compact-no-approved-data',
              'feature_mode': 'combined', 'selected_epochs': 2,
              'learning_rates': [.001, .0005], 'selection_source': 'original_A',
              'frozen_before_oof': True, 'target_units': 'native_pressure'}
    recipe_hash = residual_recipe_sha256_v007(recipe)
    records = [{'fold_id': k, 'fit_breath_ids': breath_ids[folds != k].tolist(),
                'preprocessing_fit_breath_ids': breath_ids[folds != k].tolist(),
                'base_recipe_sha256': recipe_hash,
                'checkpoint_path': f'synthetic-only-fold-{k}'} for k in range(3)]
    manifest = make_residual_oof_manifest_v007(frame, folds, records, breath_ids, forbidden, recipe)
    return {'frame': frame, 'predictions_df': predictions, 'approved_fit_breath_ids': breath_ids,
            'forbidden_breath_ids': forbidden, 'fold_ids': folds, 'base_recipe': recipe,
            'fit_records': records, 'manifest': manifest}


def run_residual_lightgbm_smoke_v007(n_jobs=2):
    """Explicit production-backend smoke on generated synthetic data only.

    Requires an existing LightGBM installation; never installs anything. Executes
    the unchanged 500-tree/127-leaf production configuration, not the toy double.
    Temporary saved artifacts are removed on exit. This does not evaluate real
    predictive quality, regenerate approved OOF base fits, or authorize training.
    """
    import platform
    import time
    from datetime import datetime, timezone
    import lightgbm

    started = time.perf_counter()
    fixture = make_residual_synthetic_fixture_v007(27, variable_residual=True)
    frame, predictions = fixture['frame'], fixture['predictions_df']
    bundle = fit_residual_v007(
        frame, predictions, fixture['manifest'], fixture['approved_fit_breath_ids'],
        fixture['forbidden_breath_ids'], model_cfg={'n_jobs': n_jobs})
    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    check('actual_LightGBM_estimator', type(bundle['model']).__module__.startswith('lightgbm'))
    check('production_fixed_capacity_unchanged',
          bundle['metadata']['model_cfg']['n_estimators'] == 500 and
          bundle['metadata']['model_cfg']['num_leaves'] == 127 and
          bundle['metadata']['model_cfg']['learning_rate'] == .03 and
          bundle['metadata']['model_cfg']['objective'] == 'regression_l1')
    check('actual_booster_fitted', bundle['model'].booster_.current_iteration() >= 1)
    y0 = predict_residual_corrected_v007(bundle, frame, predictions, 0)
    yhalf = predict_residual_corrected_v007(bundle, frame, predictions, .5, chunk_breaths=2)
    y1 = predict_residual_corrected_v007(bundle, frame, predictions, 1, chunk_breaths=3)
    base = predictions['base_pressure'].to_numpy()
    inspired = frame['u_out'].to_numpy() == 0
    native_target = frame.loc[inspired, 'pressure'].to_numpy()
    before = float(np.abs(native_target-base[inspired]).mean())
    after = float(np.abs(native_target-y1.ravel()[inspired]).mean())
    check('finite_native_prediction_breath_shape', y1.shape == (27,80) and np.isfinite(y1).all())
    check('zero_control_exact', np.array_equal(y0.ravel(), base))
    check('expiration_exact_passthrough', np.array_equal(y1.ravel()[~inspired], base[~inspired]))
    check('half_strength_linear_combination', np.allclose(yhalf, .5*(y0+y1), rtol=0, atol=1e-12))
    check('synthetic_native_error_is_learnable', after < before*.5)
    check('inspired_fit_rows_only', bundle['metadata']['fit_rows'] == int(inspired.sum()))
    with tempfile.TemporaryDirectory(prefix='v007-lightgbm-synthetic-') as directory:
        receipt = save_residual_v007(bundle, Path(directory)/'synthetic_lightgbm.pkl')
        restored = load_residual_v007(receipt['path'], expected_sha256=receipt['sha256'])
        restored_y1 = predict_residual_corrected_v007(restored, frame, predictions, 1)
        check('production_model_save_load_prediction_exact', np.array_equal(y1, restored_y1))
    return {'schema': V007_RESIDUAL_SCHEMA,
            'checked_at_utc': datetime.now(timezone.utc).isoformat(),
            'python': platform.python_version(), 'lightgbm_version': lightgbm.__version__,
            'data_scope': 'generated_synthetic_only; no assignment CSV reads or approved-data training',
            'production_lightgbm_fit_executed': True,
            'fit_rows': bundle['metadata']['fit_rows'],
            'booster_iterations': int(bundle['model'].booster_.current_iteration()),
            'synthetic_training_mae_before': before, 'synthetic_training_mae_after': after,
            'elapsed_seconds': time.perf_counter()-started,
            'checks': checks, 'check_count': len(checks), 'passed': all(checks.values())}


def run_residual_component_checks_v007():
    """Synthetic CPU checks only: no assignment CSVs, approved-data fits, or network."""
    import importlib.util
    import platform
    from datetime import datetime, timezone

    checks = {}
    def check(name, value):
        if not bool(value):
            raise AssertionError(name)
        checks[name] = True
    def rejects(name, call):
        try:
            call()
        except (ValueError, TypeError, FileExistsError):
            checks[name] = True
        else:
            raise AssertionError(name+' did not reject')

    fixture = make_residual_synthetic_fixture_v007()
    frame, predictions = fixture['frame'], fixture['predictions_df']
    breath_ids = fixture['approved_fit_breath_ids']
    forbidden, folds = fixture['forbidden_breath_ids'], fixture['fold_ids']
    recipe, records, manifest = fixture['base_recipe'], fixture['fit_records'], fixture['manifest']
    audit = validate_residual_oof_v007(frame, predictions, manifest, breath_ids, forbidden)
    check('canonical_row_ids_bound', audit['row_count'] == 720)
    check('exact_three_OOF_folds', audit['fold_count'] == 3)
    check('every_predicted_breath_excluded_from_its_base_fit', all(
        set(item['heldout_breath_ids']).isdisjoint(item['fit_breath_ids']) for item in manifest['fold_records']))
    check('preprocessing_groups_equal_actual_base_fit_groups', all(
        item['fit_breath_ids'] == item['preprocessing_fit_breath_ids'] for item in manifest['fold_records']))
    reverse = breath_ids[::-1]
    reverse_folds = residual_group_folds_v007(reverse)
    check('group_assignment_invariant_to_breath_order',
          dict(zip(breath_ids, folds)) == dict(zip(reverse, reverse_folds)))
    check('group_hash_order_invariant', residual_group_sha256_v007(breath_ids) == residual_group_sha256_v007(reverse))
    shifted = predictions.iloc[::-1].reset_index(drop=True)
    rejects('reject_misaligned_canonical_prediction_rows', lambda: validate_residual_oof_v007(frame, shifted, manifest, breath_ids, forbidden))
    contaminated = copy.deepcopy(records)
    contaminated[0]['fit_breath_ids'].append(int(breath_ids[folds == 0][0]))
    rejects('reject_base_in_sample_residuals', lambda: make_residual_oof_manifest_v007(frame, folds, contaminated, breath_ids, forbidden, recipe))
    contaminated = copy.deepcopy(records)
    contaminated[0]['preprocessing_fit_breath_ids'].append(int(breath_ids[folds == 0][0]))
    rejects('reject_heldout_preprocessing_statistics', lambda: make_residual_oof_manifest_v007(frame, folds, contaminated, breath_ids, forbidden, recipe))
    contaminated = copy.deepcopy(records)
    contaminated[0]['fit_breath_ids'].append(int(forbidden[0]))
    rejects('reject_validation_in_base_fit', lambda: make_residual_oof_manifest_v007(frame, folds, contaminated, breath_ids, forbidden, recipe))
    forbidden_overlap = np.append(forbidden, breath_ids[0])
    rejects('reject_validation_labels_in_calibration_fit', lambda: validate_residual_oof_v007(frame, predictions, manifest, breath_ids, forbidden_overlap))
    contaminated = copy.deepcopy(records)
    contaminated[0]['preprocessing_fit_breath_ids'].append(int(forbidden[0]))
    rejects('reject_validation_in_preprocessing_fit', lambda: make_residual_oof_manifest_v007(frame, folds, contaminated, breath_ids, forbidden, recipe))
    tampered = copy.deepcopy(manifest)
    tampered['fold_records'][0]['fit_group_sha256'] = '0'*64
    rejects('reject_tampered_group_hash', lambda: validate_residual_oof_v007(frame, predictions, tampered, breath_ids, forbidden))
    poisoned_recipe = dict(recipe, frozen_before_oof=False)
    rejects('reject_unfrozen_base_recipe', lambda: residual_recipe_sha256_v007(poisoned_recipe))
    rejects('reject_capacity_search_overrides', lambda: _residual_model_configuration_v007({'n_estimators': 1000}))

    model = _SyntheticMedianResidualV007()
    bundle = _fit_residual_core_v007(frame, predictions, manifest, breath_ids, forbidden,
                                   model, {'synthetic_only': True}, 'synthetic_test_double')
    inspired = frame['u_out'].to_numpy() == 0
    check('inspired_rows_only_and_native_residual_target',
          model.n_fit_rows_ == int(inspired.sum()) and np.allclose(model.target_, 2.0))
    check('expiration_NaN_labels_ignored', model.n_fit_rows_ < len(frame))
    check('known_combined47_plus_native_base_features', model.n_features_in_ == 48)
    check('IDs_and_labels_absent_from_model_features',
          not ({'id', 'breath_id', 'pressure', 'synthetic_base'} & set(bundle['metadata']['feature_names'])))
    y0 = predict_residual_corrected_v007(bundle, frame, predictions, gamma=0)
    yhalf = predict_residual_corrected_v007(bundle, frame, predictions, gamma=.5, chunk_breaths=1)
    y1 = predict_residual_corrected_v007(bundle, frame, predictions, gamma=1, chunk_breaths=2)
    base = predictions['base_pressure'].to_numpy()
    check('zero_correction_exact_control', np.array_equal(y0.ravel(), base))
    check('native_correction_and_half_strength',
          np.allclose(y1.ravel()[inspired], base[inspired]+2) and
          np.allclose(yhalf.ravel()[inspired], base[inspired]+1))
    check('expiration_exact_pass_through', np.array_equal(y1.ravel()[~inspired], base[~inspired]))
    check('output_native_breath_shape', y1.shape == (len(breath_ids), 80))
    poisoned = frame.copy()
    poisoned['pressure'] = np.inf
    poisoned['synthetic_base'] = -1e20
    check('prediction_never_reads_labels_or_unknown_columns',
          np.array_equal(y1, predict_residual_corrected_v007(bundle, poisoned, predictions, 1)))
    check('prediction_chunk_size_invariant', np.array_equal(y1, predict_residual_corrected_v007(bundle, frame, predictions, 1, chunk_breaths=9)))
    # Reorder complete groups and their explicit prediction row keys together.
    order = np.concatenate([np.arange(i*80, (i+1)*80) for i in range(8,-1,-1)])
    reordered_frame = frame.iloc[order].reset_index(drop=True)
    reordered_predictions = predictions.iloc[order].reset_index(drop=True)
    reordered = predict_residual_corrected_v007(bundle, reordered_frame, reordered_predictions, 1)
    check('group_order_equivariance', np.array_equal(reordered, y1[::-1]))
    rejects('reject_unplanned_shrinkage', lambda: predict_residual_corrected_v007(bundle, frame, predictions, .75))
    rejects('reject_prediction_row_shuffle_at_inference', lambda: predict_residual_corrected_v007(bundle, frame, shifted, 1))
    with tempfile.TemporaryDirectory(prefix='v007-residual-synthetic-') as directory:
        path = Path(directory)/'synthetic_residual.pkl'
        receipt = save_residual_v007(bundle, path)
        restored = load_residual_v007(path, expected_sha256=receipt['sha256'])
        check('save_load_prediction_exact', np.array_equal(y1, predict_residual_corrected_v007(restored, frame, predictions, 1)))
        rejects('reject_artifact_checksum_mismatch', lambda: load_residual_v007(path, expected_sha256='0'*64))
        rejects('save_protects_existing_artifact', lambda: save_residual_v007(bundle, path))
    return {
        'schema': V007_RESIDUAL_SCHEMA,
        'checked_at_utc': datetime.now(timezone.utc).isoformat(),
        'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
        'checks': checks, 'passed': all(checks.values()), 'check_count': len(checks),
        'data_scope': 'synthetic_only; no assignment CSV reads or approved-data training',
        'production_backend': 'lightgbm_fixed_MAE_127leaves_500trees',
        'production_lightgbm_available': importlib.util.find_spec('lightgbm') is not None,
        'production_lightgbm_fit_executed': False,
        'toy_backend': 'NumPy median test double for shared provenance/fitting/inference/serialization path',
        'remaining_integration': 'Run fixed production LightGBM fit in an existing course environment after genuine grouped OOF compact predictions are generated.',
    }
